## Imports

In [17]:
# Imports
import pandas as pd
import sklearn
import numpy as np

from sklearn.model_selection import cross_val_score

from sklearn.model_selection import StratifiedKFold

%load_ext autoreload
%autoreload 2
import dataprep as dp

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
# Download the csv with target
df_raw = pd.read_csv('data/train_with_target.csv')

df = df_raw.copy()

In [10]:
# add 6 week target again
df.columns

Index(['W01', 'W02', 'W03', 'W04', 'W05', 'W06', 'W07', 'W08', 'W09', 'W10',
       'W11', 'W12', 'external_code', 'season', 'category', 'release_date',
       'day', 'week', 'month', 'year', 'image_path', 'color', 'fabric',
       'extra', 'total_sales_6'],
      dtype='object')

In [21]:
df['strat_key'] = pd.qcut(df['total_sales_6'], 4, labels=False, retbins=False, precision=3, duplicates='raise')

df['strat_key'].value_counts()
df = df.reset_index(drop=True)

df.index

RangeIndex(start=0, stop=5080, step=1)

In [22]:
# Set up StratifiedKFold for validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)

fold = np.full(len(df), -1, dtype=int)

for i, (train_index, val_index) in enumerate(skf.split(df, df['strat_key'])):
    fold[val_index] = i

assert (fold == -1).sum() == 0

fold_assignment = pd.DataFrame({
    'external_code': df['external_code'],
    'fold': fold,
})
fold_assignment.to_csv('data/fold_assignment.csv', index=False)

print(fold_assignment['fold'].value_counts().sort_index())
print(fold_assignment['external_code'].nunique(), len(fold_assignment))

df = df.drop(columns=['strat_key'])

fold
0    1016
1    1016
2    1016
3    1016
4    1016
Name: count, dtype: int64
5080 5080
